# OpenStarTracker-style Bayesian Constellation Experiment

This notebook mirrors the cloned `openstartracker` BEAST flow: pair constellation database, pair hypothesis search, attitude scoring, batch tests, and FOV/magnitude accuracy reporting.


## Load Catalog


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.star_tracker_core import *

ensure_dirs()
catalog = load_catalog()
print(f"Catalog stars: {len(catalog):,}")
display(catalog.head(10))


## Catalog Plot


In [ ]:
catalog_plot_path = plot_catalog(catalog, "openstartracker_catalog_map.png")
print(catalog_plot_path)


## Build Database


In [ ]:
MAX_STARS_QUERY = 12

pair_db, db_stars = build_pair_database(
    catalog,
    algorithm_name="openstartracker",
    max_fov_deg=20.0,
    mag_limit=6.5,
)
print(f"Pair database rows: {len(pair_db):,}")
display(pair_db.head())

def identify(obs_ra, obs_dec, fov_deg, mag_limit, max_stars_query):
    return identify_openstartracker(
        catalog,
        pair_db,
        db_stars,
        obs_ra,
        obs_dec,
        fov_deg,
        mag_limit,
        max_stars_query=max_stars_query,
    )


## Single Identification Test


In [ ]:
single_result = identify(
    obs_ra=120.0,
    obs_dec=15.0,
    fov_deg=12.0,
    mag_limit=6.5,
    max_stars_query=MAX_STARS_QUERY,
)
print(single_result["outcome"])
print("stars in field:", single_result["n_stars"])
print("matched HR IDs:", single_result["matched_ids"])
print("score:", single_result["score"])
single_plot_path = plot_single_result(single_result, "openstartracker_single_result.png", "OpenStarTracker-style Bayesian single-field result")
print(single_plot_path)


## Batch Identification Test


In [ ]:
batch_results = run_batch(
    identify,
    BatchConfig(samples=80, fov_deg=10.0, mag_limit=6.5, max_stars_query=MAX_STARS_QUERY),
)
batch_summary = summarize_results(batch_results)
display(batch_results.head(10))
batch_summary


## FOV and Magnitude Accuracy Matrix


In [ ]:
FOV_VALUES = [8.0, 10.0, 12.0, 15.0, 18.0]
MAG_LIMITS = [5.0, 5.5, 6.0, 6.5, None]

confusion_df, summary_df = run_confusion_matrix(
    identify,
    fov_values=FOV_VALUES,
    mag_limits=MAG_LIMITS,
    samples=60,
    max_stars_query=MAX_STARS_QUERY,
)
confusion_path = plot_confusion_matrix(confusion_df, "OpenStarTracker-style Bayesian accuracy", "openstartracker_confusion_matrix.png")

display(confusion_df)
display(summary_df)
print(confusion_path)


## Findings


In [ ]:
report = findings(summary_df)
print(report)
